ChatGPI - API - LAB

In [53]:
#pip install datasets

In [54]:
from pathlib import Path
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd

# -----------------------------
# HELPER FUNCTIONS (NEW SECTION)
# -----------------------------

#Helper function Json Loader

import json
from pathlib import Path

def load_json_file(file_path: str) -> dict:
    """Load and parse JSON file."""
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in file {file_path}: {e}")
    
#testJson

test_data = {"test": 123}
with open("test.json", "w") as f:
    json.dump(test_data, f)

print(load_json_file("test.json"))
    
#helper function product validation

from pydantic import BaseModel, Field, ValidationError

class Product(BaseModel):
    name: str = Field(..., min_length=1)
    price: float = Field(..., gt=0)
    category: str = Field(..., min_length=1)

def validate_product_data(product_dict: dict) -> bool:
    """Validate product data using Pydantic."""
    try:
        Product(**product_dict)
        return True
    except ValidationError as e:
        print(f"Validation error: {e}")
        return False
    
#testPVal

test_data = {"name": "Test Product", "price": 99.99, "category": "Fashion"}
result = validate_product_data(test_data)
print(f"Validation result: {result}")
    
#helper function Prompt creation

def create_product_prompt(product: dict) -> str:
    
    """Generate OpenAI prompt for product."""
    
    additional_info = product.get("additional_info")
    additional_section = f"- Additional Info: {additional_info}\n" if additional_info else ""

    prompt = f"""
You are an expert e-commerce copywriter.

Product Information:
- Name: {product['name']}
- Price: ${product['price']:.2f}
- Category: {product['category']}
{additional_section}

Create:
- title (max 60 chars)
- description (150–200 words)
- features (5–7 items)
- keywords (10–15 items)

Return valid JSON only.
""".strip()

    return prompt

#test promt creation

test_product = {
    "name": "Sneakers",
    "price": 59.99,
    "category": "Fashion",
    "additional_info": "Comfortable, lightweight"
}

print(create_product_prompt(test_product))

#Helper function API Response

def parse_api_response(response_text: str) -> dict:
    #added error handling
    if not response_text:
        raise ValueError("Empty response from API")
    """Parse OpenAI API response safely."""
    try:
        parsed = json.loads(response_text)
        return parsed
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse API response: {e}")
    
#test APIresponse

test_response = '{"title": "Test", "description": "Desc", "features": [], "keywords": []}'
print(parse_api_response(test_response))
    
#helper function output formatter

def format_output(product: dict, description: dict, image_path: str) -> dict:
    """Format final output."""
    
    return {
        "product_id": product.get("id"),
        "product_name": product.get("name"),
        "price": product.get("price"),
        "category": product.get("category"),
        "image_path": image_path,
        "generated_listing": description
    }
#test output formater

product = {"id": 1, "name": "Shoes", "price": 49.99, "category": "Fashion"}
desc = {"title": "Nice shoes"}

print(format_output(product, desc, "img.jpg"))

# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
    
# Create images directory
 
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

{'test': 123}
Validation result: True
You are an expert e-commerce copywriter.

Product Information:
- Name: Sneakers
- Price: $59.99
- Category: Fashion
- Additional Info: Comfortable, lightweight


Create:
- title (max 60 chars)
- description (150–200 words)
- features (5–7 items)
- keywords (10–15 items)

Return valid JSON only.
{'title': 'Test', 'description': 'Desc', 'features': [], 'keywords': []}
{'product_id': 1, 'product_name': 'Shoes', 'price': 49.99, 'category': 'Fashion', 'image_path': 'img.jpg', 'generated_listing': {'title': 'Nice shoes'}}
Loading product dataset...
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [55]:
from pathlib import Path

images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print("Saving all images...")

for i, item in enumerate(dataset):
    try:
        image = item["image"]
        image_path = images_dir / f"product_{i}.jpg"
        image.save(image_path)
        #error handling addition
        if "image" not in item or item["image"] is None:
            raise ValueError(f"No image found for item {i}")
    except Exception as e:
        print(f"Error at image {i}: {e}")

print("Done saving.")

Saving all images...
Done saving.


In [56]:
import base64

def encode_image_to_base64(image_path):
    """
    Encodes an image file to base64 string.
    
    Args:
        image_path (str or Path): Path to the image file
        
    Returns:
        str: Base64 encoded string
    """
    try:
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
            #adder error handling
            if not encoded_string:
                raise ValueError(f"Encoding failed for image: {image_path}")
        return encoded_string
    
    except FileNotFoundError:
        print(f"❌ File not found: {image_path}")
        return None
    
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None

In [57]:
import json

def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating structured product listings.

    Parameters:
    - product_name (str): Name of the product
    - price (float): Price of the product
    - category (str): Product category
    - additional_info (str, optional): Extra product details

    Returns:
    - str: Formatted prompt
    """

    # Clean inputs
    product_name = str(product_name).strip()
    category = str(category).strip()
    additional_info = str(additional_info).strip() if additional_info else None

    # Format optional section
    additional_section = f"- Additional Info: {additional_info}\n" if additional_info else ""

    prompt = f"""
You are an expert e-commerce copywriter.

Analyze the product image together with the product metadata and generate a compelling, accurate, and professional product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{additional_section}
Instructions:
Create a product listing with the following fields:

1. title
- catchy and SEO-friendly
- maximum 60 characters

2. description
- 150 to 200 words
- persuasive but realistic
- highlight visible features, benefits, colors, materials, design elements, and likely use cases
- do not invent technical specifications that are not visible or not provided

3. features
- 5 to 7 bullet-point style items
- short, clear, and benefit-oriented

4. keywords
- 10 to 15 SEO keywords
- relevant to the product and category

Important rules:
- Base your answer only on the image and provided metadata
- If something is unclear, stay general rather than making up details
- Return valid JSON only
- Do not include markdown
- Do not include explanations before or after the JSON

Use exactly this JSON structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", "Feature 3"],
    "keywords": ["keyword1", "keyword2", "keyword3"]
}}
"""
    return prompt.strip()


# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)

print("\n" + "=" * 50)
print("PROMPT TEMPLATE")
print("=" * 50)
print(test_prompt[:1000] + "...")


PROMPT TEMPLATE
You are an expert e-commerce copywriter.

Analyze the product image together with the product metadata and generate a compelling, accurate, and professional product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Instructions:
Create a product listing with the following fields:

1. title
- catchy and SEO-friendly
- maximum 60 characters

2. description
- 150 to 200 words
- persuasive but realistic
- highlight visible features, benefits, colors, materials, design elements, and likely use cases
- do not invent technical specifications that are not visible or not provided

3. features
- 5 to 7 bullet-point style items
- short, clear, and benefit-oriented

4. keywords
- 10 to 15 SEO keywords
- relevant to the product and category

Important rules:
- Base your answer only on the image and provided metadata
- If something is unclear, stay general rather than mak

In [58]:
#pip install openai

In [59]:


import os
import json
import base64
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI


# -----------------------------
# 1. OpenAI client
# -----------------------------
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# -----------------------------
# 2. Encode image to base64
# -----------------------------
def encode_image_to_base64(image_path):
    """
    Encode an image file to a base64 string.

    Parameters:
    - image_path (str or Path): path to image

    Returns:
    - str: base64-encoded image string
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


# -----------------------------
# 3. Prompt creation
# -----------------------------
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Build the text prompt for the model.
    """
    additional_section = f"- Additional Info: {additional_info}\n" if additional_info else ""

    prompt = f"""
You are an expert e-commerce copywriter.

Analyze the product image together with the product metadata and generate a compelling, accurate, and professional product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{additional_section}
Instructions:
Create a product listing with the following fields:

1. title
- catchy and SEO-friendly
- maximum 60 characters

2. description
- 150 to 200 words
- persuasive but realistic
- highlight visible features, benefits, colors, materials, design elements, and likely use cases
- do not invent technical specifications that are not visible or not provided

3. features
- 5 to 7 items
- short, clear, and benefit-oriented

4. keywords
- 10 to 15 SEO keywords
- relevant to the product and category

Important rules:
- Base your answer only on the image and provided metadata
- If something is unclear, stay general rather than making up details
- Return the result in the requested JSON structure only
""".strip()

    return prompt


# -----------------------------
# 4. JSON schema for strict output
# -----------------------------
PRODUCT_LISTING_SCHEMA = {
    "name": "product_listing",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string"
            },
            "description": {
                "type": "string"
            },
            "features": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 5,
                "maxItems": 7
            },
            "keywords": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 10,
                "maxItems": 15
            }
        },
        "required": ["title", "description", "features", "keywords"],
        "additionalProperties": False
    }
}


# -----------------------------
# 5. Main API call before refact
# -----------------------------
""""def generate_product_listing(image_path, product_name, price, category, additional_info=None, model="gpt-4.1"):
    
    Send image + prompt to OpenAI and return parsed JSON output.

    Parameters:
    - image_path (str or Path)
    - product_name (str)
    - price (float)
    - category (str)
    - additional_info (str, optional)
    - model (str): OpenAI model name

    Returns:
    - dict: parsed JSON product listing

    prompt = create_product_listing_prompt(
        product_name=product_name,
        price=price,
        category=category,
        additional_info=additional_info
    )

    base64_image = encode_image(image_path)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    }
                ]
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": PRODUCT_LISTING_SCHEMA["name"],
                "strict": PRODUCT_LISTING_SCHEMA["strict"],
                "schema": PRODUCT_LISTING_SCHEMA["schema"]
            }
        }
    )

    # The API returns text output; with structured output requested, this should be valid JSON
    raw_text = response.output_text
    parsed = json.loads(raw_text)

    return parsed
    """

#----
#5. Main API call after refact

from pathlib import Path
import json

# -----------------------------
# 1. Build API request
# -----------------------------
def build_api_request(image_path, product_data):
    prompt = create_product_prompt(product_data)
    base64_image = encode_image_to_base64(image_path)
    if not base64_image:
        raise ValueError(f"Failed to encode image: {image_path}")
    return prompt, base64_image


# -----------------------------
# 2. Call OpenAI API
# -----------------------------
def call_openai_api(client, prompt, base64_image, schema, model="gpt-4.1"):

    try:
        response = client.responses.create(
            model=model,
            input=[
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": prompt},
                        {
                            "type": "input_image",
                            "image_url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "high"
                        }
                    ]
                }
            ],
            text={
                "format": {
                    "type": "json_schema",
                    "name": schema["name"],
                    "strict": schema["strict"],
                    "schema": schema["schema"]
                }
            }
        )
    except Exception as e:
        raise RuntimeError(f"OpenAI API call failed: {e}")

    if not hasattr(response, "output_text") or not response.output_text:
        raise ValueError("Invalid API response structure")

    return response.output_text

# -----------------------------
# 3. Main orchestration function
# -----------------------------
def generate_product_listing(image_path, product_data, client, schema):

    if not validate_product_data(product_data):
        raise ValueError("Invalid product data")

    prompt, base64_image = build_api_request(image_path, product_data)

    response_text = call_openai_api(client, prompt, base64_image, schema)

    parsed = parse_api_response(response_text)

    return parsed


# -----------------------------
# 4. Prepare test data
# -----------------------------
product_data = {
    "name": "Running Shoes",
    "price": 89.99,
    "category": "Sportswear",
    "additional_info": "Lightweight, breathable mesh"
}

images_dir = Path("product_images")
image_files = list(images_dir.glob("*.jpg"))

if not image_files:
    raise ValueError("No images found in product_images folder")

image_path = image_files[0]

print("Using image:", image_path)


# -----------------------------
# 5. Test run 
# -----------------------------
try:
    result = generate_product_listing(
        image_path=image_path,
        product_data=product_data,
        client=client,
        schema=PRODUCT_LISTING_SCHEMA
    )

    print("✓ API call succeeded")
    print("✓ JSON parsed correctly\n")
    print(json.dumps(result, indent=4))

except FileNotFoundError as e:
    print(f"Image error: {e}")

except json.JSONDecodeError as e:
    print(f"JSON parsing error: {e}")

except Exception as e:
    print(f"API/request error: {e}")

Using image: product_images/product_76.jpg
✓ API call succeeded
✓ JSON parsed correctly

{
    "title": "Lightweight Breathable Running Shoes for Active Lifestyles",
    "description": "Experience the perfect blend of comfort, style, and performance with our Running Shoes, designed for athletes and enthusiasts alike. Crafted from high-quality, lightweight materials and featuring a breathable mesh upper, these shoes provide superior ventilation to keep your feet cool and dry during even the most intense workouts. The ergonomic sole offers excellent cushioning and shock absorption, ensuring a smooth ride whether you're hitting the track, pavement, or gym. The sleek design is versatile enough for casual wear while offering the advanced support needed for long-distance runs. Choose our Running Shoes and elevate your athletic performance with footwear engineered for those who demand both quality and comfort.",
    "features": [
        "Ultra-lightweight construction for enhanced speed",
  

PArt 6

In [60]:
#version prior to refactoring
"""import time
import pandas as pd
import json
from pathlib import Path


def process_multiple_products(dataset, max_products=5, output_file="generated_product_listings.json"):

    Generate product listings for multiple products and save results.


    results = []
    errors = []

    for i in range(max_products):
        print(f"\nProcessing product {i + 1}/{max_products}...")

        try:
            product = dataset[i]

            image_path = f"product_images/product_{i}.jpg"
            product_name = product.get("productDisplayName", f"Fashion Product {i}")
            category = product.get("articleType", "Fashion")
            price = 49.99

            additional_info = (
                f"Gender: {product.get('gender', 'Unknown')}, "
                f"Category: {product.get('masterCategory', 'Unknown')}, "
                f"Subcategory: {product.get('subCategory', 'Unknown')}, "
                f"Color: {product.get('baseColour', 'Unknown')}, "
                f"Season: {product.get('season', 'Unknown')}, "
                f"Usage: {product.get('usage', 'Unknown')}"
            )

            listing = generate_product_listing(
                image_path=image_path,
                product_name=product_name,
                price=price,
                category=category,
                additional_info=additional_info
            )

            result = {
                "product_id": product.get("id", i),
                "product_name": product_name,
                "price": price,
                "category": category,
                "image_path": image_path,
                "generated_listing": listing
            }

            results.append(result)
            print(f"✓ Success: {product_name}")

            # small pause to avoid sending requests too quickly
            time.sleep(1)

        except Exception as e:
            error_info = {
                "product_index": i,
                "error": str(e)
            }
            errors.append(error_info)
            print(f"⚠ Error processing product {i}: {e}")

    # Save successful results
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    # Save errors separately
    with open("product_listing_errors.json", "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=4, ensure_ascii=False)

    print("\nBatch processing complete!")
    print(f"Successful listings: {len(results)}")
    print(f"Errors: {len(errors)}")
    print(f"Saved results to: {output_file}")
    print("Saved errors to: product_listing_errors.json")

    return results, errors
    """

'import time\nimport pandas as pd\nimport json\nfrom pathlib import Path\n\n\ndef process_multiple_products(dataset, max_products=5, output_file="generated_product_listings.json"):\n\n    Generate product listings for multiple products and save results.\n\n\n    results = []\n    errors = []\n\n    for i in range(max_products):\n        print(f"\nProcessing product {i + 1}/{max_products}...")\n\n        try:\n            product = dataset[i]\n\n            image_path = f"product_images/product_{i}.jpg"\n            product_name = product.get("productDisplayName", f"Fashion Product {i}")\n            category = product.get("articleType", "Fashion")\n            price = 49.99\n\n            additional_info = (\n                f"Gender: {product.get(\'gender\', \'Unknown\')}, "\n                f"Category: {product.get(\'masterCategory\', \'Unknown\')}, "\n                f"Subcategory: {product.get(\'subCategory\', \'Unknown\')}, "\n                f"Color: {product.get(\'baseColour\', 

In [ ]:
#previous non-refracted version for results, errors = process_multiple_products(
  """ dataset=dataset,
    max_products=5,
    output_file="generated_product_listings.json"
)
import json

with open("generated_product_listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("✅ JSON file saved successfully!")"""

IndentationError: unexpected indent (2493539253.py, line 2)

In [ ]:
import time
import json
from pathlib import Path


def process_multiple_products(
    dataset,
    max_products=5,
    output_file="generated_product_listings.json"
):
    """
    Generate product listings for multiple products using refactored pipeline.
    """

    results = []
    errors = []

    images_dir = Path("product_images")

    for i in range(min(max_products, len(dataset))):
        print(f"\nProcessing product {i + 1}/{max_products}...")

        try:
            product = dataset[i]

            # -----------------------------
            # 1. Build image path
            # -----------------------------
            image_path = images_dir / f"product_{i}.jpg"

            if not image_path.exists():
                raise FileNotFoundError(f"Image not found: {image_path}")

            # -----------------------------
            # 2. Build product_data (NEW STRUCTURE)
            # -----------------------------
            product_data = {
                "name": product.get("productDisplayName", f"Fashion Product {i}"),
                "price": 49.99,
                "category": product.get("articleType", "Fashion"),
                "additional_info": (
                    f"Gender: {product.get('gender', 'Unknown')}, "
                    f"Category: {product.get('masterCategory', 'Unknown')}, "
                    f"Subcategory: {product.get('subCategory', 'Unknown')}, "
                    f"Color: {product.get('baseColour', 'Unknown')}, "
                    f"Season: {product.get('season', 'Unknown')}, "
                    f"Usage: {product.get('usage', 'Unknown')}"
                )
            }

            # -----------------------------
            # 3. Generate listing (NEW API)
            # -----------------------------
            listing = generate_product_listing(
                image_path=image_path,
                product_data=product_data,
                client=client,
                schema=PRODUCT_LISTING_SCHEMA
            )

            # -----------------------------
            # 4. Format output (USE HELPER)
            # -----------------------------
            result = format_output(product, listing, str(image_path))

            results.append(result)

            print(f"✓ Success: {product_data['name']}")

            # Avoid rate limits
            time.sleep(1)

        except Exception as e:
            error_info = {
                "product_index": i,
                "error": str(e)
            }

            errors.append(error_info)
            print(f"⚠ Error processing product {i}: {e}")

    # -----------------------------
    # 5. Save results
    # -----------------------------
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    with open("product_listing_errors.json", "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=4, ensure_ascii=False)

    print("\nBatch processing complete!")
    print(f"Successful listings: {len(results)}")
    print(f"Errors: {len(errors)}")
    print(f"Saved results to: {output_file}")

    return results, errors

results, errors = process_multiple_products(
    dataset=dataset,
    max_products=10,
    output_file="generated_product_listings.json")


Processing product 1/10...
✓ Success: Turtle Check Men Navy Blue Shirt

Processing product 2/10...
✓ Success: Peter England Men Party Blue Jeans

Processing product 3/10...
✓ Success: Titan Women Silver Watch

Processing product 4/10...
✓ Success: Manchester United Men Solid Black Track Pants

Processing product 5/10...
✓ Success: Puma Men Grey T-shirt

Processing product 6/10...
✓ Success: Inkfruit Mens Chain Reaction T-shirt

Processing product 7/10...
✓ Success: Fabindia Men Striped Green Shirt

Processing product 8/10...
✓ Success: Jealous 21 Women Purple Shirt

Processing product 9/10...
✓ Success: Puma Men Pack of 3 Socks

Processing product 10/10...
✓ Success: Skagen Men Black Watch

Batch processing complete!
Successful listings: 10
Errors: 0
Saved results to: generated_product_listings.json


Error handling: Well, where should i start. This felt like an excersice in debugging at the end. I was getting a lot of errors. Fortunately, most of them were related to the renaming of sections. So files were mostly not found. I obviously also forgot to change to the API key, so this occured too and I was unable to call the model. Formating was also various time wrong, but I am getting a good feeling on how to improve these errors. Some of the errors are still visible above. Note: I am sorry that I did not handle the errors when they ocurred.  

orignal Report: So basically, the way the API works in this project is pretty straightforward.
I take an image of a product and some basic information like the name, price, and category. Then I convert the image into a format the API understands (base64), and send both the image and the text prompt to the OpenAI API.

The API then analyzes the image together with the prompt and generates a full product listing — including a title, description, features, and keywords. It sends the result back as a JSON response, which I then parse in Python and store for later use.

I definitely ran into quite a few challenges along the way.

At the beginning, I had trouble loading and handling the dataset properly. The images were not automatically saved, so I had to figure out how to extract them from the Hugging Face dataset and store them locally.

Another big issue was with the API setup. I had created an .env file, but I named it incorrectly, so the API key wasn’t being recognized. That took a bit of time to debug, but once I fixed the environment variable loading, the API started working.

Towards the end, I also had difficulties generating and saving the JSON files correctly. Sometimes variables were not defined, or the file wasn’t being created properly, so I had to re-run parts of the pipeline and make sure everything was executed in the right order.

Once everything was working, the quality of the generated product listings was actually good.